# Phoneme CTC fine-tuning (pilot): Hafs2Vec recipe

Fine-tunes XLS-R with CTC on `phoneme_ref`, following Hafs2Vec (Ibrahim, 2025). Run the cells top to bottom.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine for the 300m model).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install
Versions are pinned to the ones the script was tested with.

In [ ]:
!pip install -q transformers==5.21.0 datasets==5.1.0 accelerate==1.15.0 soundfile

## 2. Mount Google Drive (to keep the results)
Training writes to Colab's local disk, which is fast and is wiped when the session ends. After training, a cell copies only the final model and results to Drive (about 1.3 GB). Do not train straight into Drive: checkpoints pile up in Drive's cache and Trash and fill the disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Write the training script and phoneme vocabulary
Generated from `training/train_ctc.py` and `training/vocab.json` in the repo; no need to edit.

In [ ]:
%%writefile train_ctc.py
#!/usr/bin/env python3
"""Fine-tune wav2vec2 / XLS-R with CTC on phoneme sequences (`phoneme_ref`).

Follows the recipe of Hafs2Vec (Ibrahim, 2025, ArabicNLP): XLS-R + CTC on Tajweed-aware phonemes from the
Quranic Phonemizer, AdamW, linear warm-up (ratio 0.1), mixed precision, greedy decoding, evaluated with PER
broken into substitutions / deletions / insertions. Differences for this pilot are listed in the README.

Typical Colab run:
    python train_ctc.py --dataset <user>/quran-tajweed-iqlab-pilot --output_dir /content/run1

Audio is decoded with soundfile from the stored WAV bytes, so torchcodec is not needed.
"""
import argparse, csv, io, json, os, random, sys
from pathlib import Path

import numpy as np

BLANK, UNK = "<blank>", "<unk>"


# --------------------------------------------------------------------------- PER with S/D/I
def align(ref, hyp):
    """Levenshtein alignment of two token lists. Returns (ops, S, D, I) where ops is a list of
    (op, ref_token, hyp_token) with op in {'=', 'S', 'D', 'I'}."""
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), dtype=np.int32)
    d[:, 0] = np.arange(n + 1)
    d[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("=" if ref[i - 1] == hyp[j - 1] else "S", ref[i - 1], hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            ops.append(("D", ref[i - 1], None)); i -= 1
        else:
            ops.append(("I", None, hyp[j - 1])); j -= 1
    ops.reverse()
    S = sum(o[0] == "S" for o in ops); D = sum(o[0] == "D" for o in ops); I = sum(o[0] == "I" for o in ops)
    return ops, S, D, I


def per_report(refs, hyps, focus=()):
    """Corpus-level PER (%) with substitution / deletion / insertion rates, plus per-token accuracy for
    `focus` tokens (how often a reference focus phoneme was recognised as itself)."""
    N = S = D = I = 0
    hit = {t: 0 for t in focus}; tot = {t: 0 for t in focus}
    for r, h in zip(refs, hyps):
        ops, s, dl, ins = align(r, h)
        N += len(r); S += s; D += dl; I += ins
        for op, rt, ht in ops:
            if rt in tot:
                tot[rt] += 1
                hit[rt] += (op == "=")
    pct = lambda x: round(100.0 * x / max(N, 1), 2)
    out = {"per": pct(S + D + I), "sub": pct(S), "del": pct(D), "ins": pct(I), "ref_phonemes": N}
    for t in focus:
        out[f"acc[{t}]"] = round(100.0 * hit[t] / tot[t], 2) if tot[t] else None
        out[f"count[{t}]"] = tot[t]
    return out


# --------------------------------------------------------------------------- vocab
def load_or_build_vocab(path, texts):
    if path and Path(path).exists():
        vocab = json.load(open(path, encoding="utf-8"))
    else:
        vocab = {BLANK: 0, UNK: 1}
        for t in sorted({t for s in texts for t in s.split()}):
            vocab[t] = len(vocab)
    assert vocab.get(BLANK) == 0, "blank must have id 0"
    unseen = sorted({t for s in texts for t in s.split()} - set(vocab))
    if unseen:
        print(f"warning: {len(unseen)} phoneme(s) not in vocab, mapped to {UNK}: {unseen[:10]}")
    return vocab


def encode(text, vocab):
    unk = vocab[UNK]
    return [vocab.get(t, unk) for t in text.split()]


def ctc_greedy(ids, inv, blank=0):
    """Collapse repeats, drop blanks. `ids` may contain -100 padding (treated as blank)."""
    out, prev = [], None
    for i in ids:
        i = int(i)
        if i < 0:
            i = blank
        if i != prev and i != blank:
            out.append(inv.get(i, UNK))
        prev = i
    return out


# --------------------------------------------------------------------------- audio
def decode_audio(a):
    import soundfile as sf
    if a.get("bytes"):
        wav, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
    else:
        wav, sr = sf.read(a["path"], dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != 16000:
        raise ValueError(f"expected 16 kHz audio, got {sr} Hz")
    return wav


class Collator:
    def __init__(self, fe, vocab):
        self.fe, self.vocab = fe, vocab

    def __call__(self, rows):
        import torch
        audio = [decode_audio(r["audio"]) for r in rows]
        batch = self.fe(audio, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors="pt")
        labs = [encode(r["phoneme_ref"], self.vocab) for r in rows]
        L = max(len(x) for x in labs)
        lab = torch.full((len(labs), L), -100, dtype=torch.long)
        for k, x in enumerate(labs):
            lab[k, : len(x)] = torch.tensor(x, dtype=torch.long)
        batch["labels"] = lab
        return batch


# --------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", required=True, help="HF dataset id, or a folder saved with save_to_disk")
    ap.add_argument("--model", default="facebook/wav2vec2-xls-r-300m",
                    help="pretrained checkpoint. Hafs2Vec used facebook/wav2vec2-xls-r-1b (needs an A100-class GPU)")
    ap.add_argument("--vocab", default=str(Path(__file__).with_name("vocab.json")))
    ap.add_argument("--output_dir", default="run")
    ap.add_argument("--train_split", default="train")
    ap.add_argument("--eval_split", default="test")
    ap.add_argument("--epochs", type=float, default=60, help="Hafs2Vec: 15 (on ~170 h of data)")
    ap.add_argument("--max_steps", type=int, default=-1, help="overrides --epochs when > 0 (for smoke tests)")
    ap.add_argument("--lr", type=float, default=1e-4, help="Hafs2Vec: 3e-5 (much more data than this pilot)")
    ap.add_argument("--warmup_ratio", type=float, default=0.1)
    ap.add_argument("--batch_size", type=int, default=2, help="per device; long ayahs (up to ~50 s) need small batches on a T4")
    ap.add_argument("--grad_accum", type=int, default=4, help="effective batch = batch_size x grad_accum (Hafs2Vec: 352)")
    ap.add_argument("--max_duration", type=float, default=60.0,
                    help="drop clips longer than this many seconds (Hafs2Vec: 10)")
    ap.add_argument("--focus", nargs="*", default=["m̃"], help="phonemes to report recognition accuracy for")
    ap.add_argument("--select_best", choices=["loss", "per", "last"], default="loss",
                    help="which checkpoint becomes the final model: lowest eval loss (default), lowest eval PER "
                         "(ties keep the EARLIER epoch, so a run stuck at PER 100 would keep epoch 1), or the last epoch")
    ap.add_argument("--no_gradient_checkpointing", action="store_true")
    ap.add_argument("--hub_model_id", default=None, help="push the final model to this HF repo (optional)")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    import torch
    from datasets import load_dataset, load_from_disk, Audio
    from transformers import (Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC, Trainer, TrainingArguments,
                              set_seed)
    set_seed(a.seed); random.seed(a.seed)

    dd = load_from_disk(a.dataset) if os.path.isdir(a.dataset) else load_dataset(a.dataset)
    for s in (a.train_split, a.eval_split):
        if s not in dd:
            sys.exit(f"split '{s}' not in dataset (have: {list(dd)})")
    dd = dd.cast_column("audio", Audio(decode=False))

    def keep(ex):
        return ex["duration_s"] is None or ex["duration_s"] <= a.max_duration
    for s in (a.train_split, a.eval_split):
        before = len(dd[s]); dd[s] = dd[s].filter(keep)
        print(f"{s}: {len(dd[s])}/{before} clips <= {a.max_duration}s, "
              f"{sum(list(dd[s]['duration_s'])) / 60:.1f} min, speakers={sorted(set(list(dd[s]['speaker_id'])))}")
        if len(dd[s]) == 0:
            sys.exit(f"no clips left in '{s}'; raise --max_duration")

    vocab = load_or_build_vocab(a.vocab, list(dd[a.train_split]["phoneme_ref"]) + list(dd[a.eval_split]["phoneme_ref"]))
    inv = {i: t for t, i in vocab.items()}
    print(f"vocab: {len(vocab)} symbols")

    fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0,
                                  do_normalize=True, return_attention_mask=True)
    model = Wav2Vec2ForCTC.from_pretrained(
        a.model, vocab_size=len(vocab), pad_token_id=vocab[BLANK],
        ctc_loss_reduction="mean", ctc_zero_infinity=True, ignore_mismatched_sizes=True)
    model.freeze_feature_encoder()
    if not a.no_gradient_checkpointing:
        model.gradient_checkpointing_enable()

    def argmax_logits(logits, labels):
        return logits.argmax(-1)

    def metrics(p):
        pred_ids, label_ids = p.predictions, p.label_ids
        hyps = [ctc_greedy(x, inv) for x in pred_ids]
        refs = [[inv[int(i)] for i in row if i != -100] for row in label_ids]
        return per_report(refs, hyps, a.focus)

    cuda = torch.cuda.is_available()
    # bf16 only on GPUs with native support (compute capability 8+: A100, L4, H100). On a T4 (7.5),
    # torch.cuda.is_bf16_supported() can report emulated support, which is slower than fp16.
    bf16 = cuda and torch.cuda.get_device_capability()[0] >= 8
    import inspect
    accepted = inspect.signature(TrainingArguments.__init__).parameters
    kw = dict(
        output_dir=a.output_dir, per_device_train_batch_size=a.batch_size, per_device_eval_batch_size=a.batch_size,
        gradient_accumulation_steps=a.grad_accum, learning_rate=a.lr,
        num_train_epochs=a.epochs, max_steps=a.max_steps, optim="adamw_torch", lr_scheduler_type="linear",
        fp16=cuda and not bf16, bf16=bf16, save_total_limit=1,
        logging_steps=5, report_to="none", remove_unused_columns=False, dataloader_num_workers=2 if cuda else 0,
        seed=a.seed)
    if a.select_best == "last":
        kw.update(save_strategy="no", load_best_model_at_end=False)       # no checkpoints; keep the final weights
    else:
        kw.update(save_strategy="epoch", load_best_model_at_end=True, greater_is_better=False,
                  metric_for_best_model=a.select_best)                   # "loss" -> eval_loss, "per" -> eval_per
    # keyword names differ across transformers versions
    kw["eval_strategy" if "eval_strategy" in accepted else "evaluation_strategy"] = "epoch"
    if "save_only_model" in accepted:
        kw["save_only_model"] = True                 # checkpoints hold weights only (~1.3 GB for 300m, not ~3.8 GB)
    if "warmup_ratio" in accepted:
        kw["warmup_ratio"] = a.warmup_ratio          # transformers 4.x
    else:
        kw["warmup_steps"] = a.warmup_ratio          # 5.x: a float in [0, 1) is a ratio
    args = TrainingArguments(**kw)
    trainer = Trainer(model=model, args=args, train_dataset=dd[a.train_split], eval_dataset=dd[a.eval_split],
                      data_collator=Collator(fe, vocab), compute_metrics=metrics,
                      preprocess_logits_for_metrics=argmax_logits)
    print(f"device: {torch.cuda.get_device_name(0) if cuda else 'cpu'}, precision: {'bf16' if bf16 else 'fp16' if cuda else 'fp32'}")
    trainer.train()

    # final evaluation with per-utterance predictions
    out = Path(a.output_dir); out.mkdir(parents=True, exist_ok=True)
    pred = trainer.predict(dd[a.eval_split])
    hyps = [ctc_greedy(x, inv) for x in pred.predictions]
    ev = dd[a.eval_split]
    refs = [r.split() for r in list(ev["phoneme_ref"])]
    summary = per_report(refs, hyps, a.focus)
    summary.update({"model": a.model, "train_clips": len(dd[a.train_split]), "eval_clips": len(ev),
                    "eval_split": a.eval_split, "selected_by": a.select_best,
                    "selected_checkpoint": trainer.state.best_model_checkpoint})
    with open(out / "predictions.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "speaker_id", "surah", "ayah", "per", "sub", "del", "ins", "phoneme_ref", "phoneme_pred"])
        for k in range(len(ev)):
            _, s, d_, i_ = align(refs[k], hyps[k])
            w.writerow([ev[k]["id"], ev[k]["speaker_id"], ev[k]["surah"], ev[k]["ayah"],
                        round(100 * (s + d_ + i_) / max(len(refs[k]), 1), 2), s, d_, i_,
                        " ".join(refs[k]), " ".join(hyps[k])])
    json.dump(summary, open(out / "summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("\nFINAL", json.dumps(summary, ensure_ascii=False))

    final = out / "final"
    trainer.save_model(str(final)); fe.save_pretrained(str(final))
    json.dump(vocab, open(final / "vocab.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    import shutil
    for ck in out.glob("checkpoint-*"):              # the final model is saved; drop intermediate checkpoints
        shutil.rmtree(ck, ignore_errors=True)
    print(f"model, vocab, predictions.csv and summary.json saved in {out}")
    if a.hub_model_id:
        model.push_to_hub(a.hub_model_id, private=True); fe.push_to_hub(a.hub_model_id, private=True)
        from huggingface_hub import upload_file
        upload_file(path_or_fileobj=str(final / "vocab.json"), path_in_repo="vocab.json", repo_id=a.hub_model_id)
        print(f"pushed to https://huggingface.co/{a.hub_model_id}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile vocab.json
{
 "<blank>": 0,
 "<unk>": 1,
 "Q": 2,
 "a": 3,
 "a:": 4,
 "aˤ:": 5,
 "b": 6,
 "bb": 7,
 "d": 8,
 "dd": 9,
 "dˤ": 10,
 "dˤdˤ": 11,
 "f": 12,
 "ff": 13,
 "h": 14,
 "hh": 15,
 "i": 16,
 "i:": 17,
 "j": 18,
 "jj": 19,
 "j̃": 20,
 "k": 21,
 "kk": 22,
 "l": 23,
 "ll": 24,
 "lˤlˤ": 25,
 "m": 26,
 "m̃": 27,
 "n": 28,
 "q": 29,
 "qq": 30,
 "r": 31,
 "rr": 32,
 "rˤ": 33,
 "rˤrˤ": 34,
 "s": 35,
 "ss": 36,
 "sˤ": 37,
 "sˤsˤ": 38,
 "t": 39,
 "tt": 40,
 "tˤ": 41,
 "tˤtˤ": 42,
 "u": 43,
 "u:": 44,
 "w": 45,
 "ww": 46,
 "w̃": 47,
 "x": 48,
 "xx": 49,
 "z": 50,
 "zz": 51,
 "ð": 52,
 "ðð": 53,
 "ðˤ": 54,
 "ðˤðˤ": 55,
 "ñ": 56,
 "ħ": 57,
 "ħħ": 58,
 "ŋ": 59,
 "ɣ": 60,
 "ʃ": 61,
 "ʃʃ": 62,
 "ʒ": 63,
 "ʒʒ": 64,
 "ʔ": 65,
 "ʕ": 66,
 "ʕʕ": 67,
 "θ": 68,
 "θθ": 69
}

In [ ]:
%%writefile evaluate_ctc.py
#!/usr/bin/env python3
"""Evaluate an already-trained CTC model on any split, without training.

    python evaluate_ctc.py --model_dir run/final --dataset <user>/quran-tajweed-iqlab-pilot --split train

Writes predictions_<split>.csv and summary_<split>.json next to the model folder (or in --output_dir).
Uses the same decoding and scoring as train_ctc.py, so the numbers are directly comparable.
"""
import argparse, csv, json, os, sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
from train_ctc import Collator, align, ctc_greedy, per_report  # noqa: E402


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model_dir", required=True, help="the final/ folder written by train_ctc.py")
    ap.add_argument("--dataset", required=True, help="HF dataset id, or a folder saved with save_to_disk")
    ap.add_argument("--split", default="train", help="split to score, e.g. train or test")
    ap.add_argument("--output_dir", default=None, help="default: the folder that contains --model_dir")
    ap.add_argument("--batch_size", type=int, default=4)
    ap.add_argument("--focus", nargs="*", default=["m̃"])
    a = ap.parse_args()

    import torch
    from datasets import load_dataset, load_from_disk, Audio
    from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC

    md = Path(a.model_dir)
    vocab = json.load(open(md / "vocab.json", encoding="utf-8"))
    inv = {i: t for t, i in vocab.items()}
    fe = Wav2Vec2FeatureExtractor.from_pretrained(str(md))
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = Wav2Vec2ForCTC.from_pretrained(str(md)).to(dev).eval()

    dd = load_from_disk(a.dataset) if os.path.isdir(a.dataset) else load_dataset(a.dataset)
    if a.split not in dd:
        sys.exit(f"split '{a.split}' not in dataset (have: {list(dd)})")
    ds = dd[a.split].cast_column("audio", Audio(decode=False))
    collate = Collator(fe, vocab)
    print(f"scoring {len(ds)} clips from '{a.split}' with {md} on {dev}")

    refs, hyps, rows = [], [], []
    for start in range(0, len(ds), a.batch_size):
        chunk = [ds[k] for k in range(start, min(start + a.batch_size, len(ds)))]
        batch = collate(chunk)
        with torch.no_grad():
            logits = model(input_values=batch["input_values"].to(dev),
                           attention_mask=batch["attention_mask"].to(dev)).logits
        for r, ids in zip(chunk, logits.argmax(-1).cpu().tolist()):
            ref, hyp = r["phoneme_ref"].split(), ctc_greedy(ids, inv)
            refs.append(ref); hyps.append(hyp); rows.append(r)

    summary = per_report(refs, hyps, a.focus)
    summary.update({"model_dir": str(md), "split": a.split, "clips": len(rows)})
    out = Path(a.output_dir) if a.output_dir else md.parent
    out.mkdir(parents=True, exist_ok=True)
    with open(out / f"predictions_{a.split}.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "speaker_id", "surah", "ayah", "per", "sub", "del", "ins", "phoneme_ref", "phoneme_pred"])
        for r, ref, hyp in zip(rows, refs, hyps):
            _, s, d, i = align(ref, hyp)
            w.writerow([r["id"], r["speaker_id"], r["surah"], r["ayah"], round(100 * (s + d + i) / max(len(ref), 1), 2),
                        s, d, i, " ".join(ref), " ".join(hyp)])
    json.dump(summary, open(out / f"summary_{a.split}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("RESULT", json.dumps(summary, ensure_ascii=False))
    print(f"wrote {out / f'predictions_{a.split}.csv'} and {out / f'summary_{a.split}.json'}")


if __name__ == "__main__":
    main()

## 4. Settings
Set `DATASET` to your Hugging Face dataset. Model choice:
- `facebook/wav2vec2-xls-r-300m`: fits a free **T4** (default).
- `facebook/wav2vec2-xls-r-1b`: what Hafs2Vec used; needs an **A100** (Colab Pro) and smaller batches.

In [ ]:
import os
DATASET = "YOUR_HF_USERNAME/quran-tajweed-iqlab-pilot"   # <- change this
MODEL = "facebook/wav2vec2-xls-r-300m"
RUN_NAME = "iqlab_pilot_300m"
OUTPUT_DIR = f"/content/{RUN_NAME}"                       # local disk while training
DRIVE_DIR = f"/content/drive/MyDrive/thesis_runs/{RUN_NAME}"  # where the results are copied afterwards
EPOCHS, LR, BATCH, ACCUM = 60, 1e-4, 2, 4
print(OUTPUT_DIR, "->", DRIVE_DIR)

In [ ]:
!df -h /content | tail -1   # free disk before training

## 5. Train
Prints evaluation (PER, sub/del/ins, accuracy on `m̃`) after every epoch on the held-out reciter.

In [ ]:
!python train_ctc.py --dataset "{DATASET}" --model "{MODEL}" --output_dir "{OUTPUT_DIR}" --epochs {EPOCHS} --lr {LR} --batch_size {BATCH} --grad_accum {ACCUM}

## 6. Copy the results to Drive
Copies `final/` (the model), `predictions.csv` and `summary.json`. Checkpoints are not copied.

In [ ]:
import os, shutil
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for name in ["summary.json", "predictions.csv"]:
        shutil.copy(f"{OUTPUT_DIR}/{name}", DRIVE_DIR)
    shutil.copytree(f"{OUTPUT_DIR}/final", f"{DRIVE_DIR}/final", dirs_exist_ok=True)
    print("copied to", DRIVE_DIR)
else:
    print("Drive is not mounted; results are only in", OUTPUT_DIR)

## 7. Results

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f"{OUTPUT_DIR}/summary.json")), indent=1, ensure_ascii=False))
pd.set_option("display.max_colwidth", 120)
pd.read_csv(f"{OUTPUT_DIR}/predictions.csv")[["id", "per", "sub", "del", "ins", "phoneme_ref", "phoneme_pred"]]

## 8. Did the model learn anything? Evaluate on the training reciters
No retraining: this scores the model you just trained on the `train` split (Alafasy, Husary, Minshawy).
If training PER is well below 100% while test PER is near 100%, the model learned but does not yet
generalise to a new voice. If both are near 100%, it is still in the blank phase: train longer or add data.
Works after a restart too: point `MODEL_DIR` at `DRIVE_DIR + "/final"`.

In [ ]:
MODEL_DIR = f"{OUTPUT_DIR}/final" if os.path.isdir(f"{OUTPUT_DIR}/final") else f"{DRIVE_DIR}/final"
!python evaluate_ctc.py --model_dir "{MODEL_DIR}" --dataset "{DATASET}" --split train --output_dir "{OUTPUT_DIR}"
pd.read_csv(f"{OUTPUT_DIR}/predictions_train.csv")[["id", "per", "sub", "del", "ins", "phoneme_ref", "phoneme_pred"]]